In [ ]:
#libraries
library(tidyverse)
library(readxl)
library(here)
library(openxlsx)
library(writexl)

## Reading the data

#### Dataset: Pupils.xlxs

Data cleaning: 
- Column names converted to lowercase with underscores.
- `date_of_birth` and `start_date` were converted to dates.
- `current_year_group` was stored as text and converted to integer.
- Final data: 163 pupils, one row per pupil.

In [25]:
# Pupils.xlxs
pupils_raw <- read_excel(here("data/raw/Pupils.xlsx"), sheet = "Export")

glimpse(pupils_raw)

Rows: 163
Columns: 6
$ `Pupil ID`           <chr> "Pupil 191", "Pupil 192", "Pupil 193", "Pupil 194…
$ `Date of birth`      <dttm> 2022-06-02, 2022-07-06, 2022-08-11, 2022-06-14, …
$ `Current class`      <chr> "Starter group", "Starter group", "Starter group"…
$ `Current year group` <chr> "0", "0", "0", "0", "0", "0", "0", "0", "0", "0",…
$ Pupil                <chr> "Noor Verbeek", "Daan Veldman", "Isa Van Loon", "…
$ `Start date`         <dttm> 2026-06-02, 2026-07-06, 2026-08-31, 2026-06-17, …


In [27]:
# clean the pupils data
pupils <- pupils_raw |>
  # Rename columns to short names without spaces
  rename(
    pupil_id = `Pupil ID`,
    date_of_birth = `Date of birth`,
    current_class = `Current class`,
    current_year_group = `Current year group`,
    pupil_name = Pupil,
    start_date  = `Start date`
  ) |>
  # transform year group into number
  mutate(current_year_group = as.integer(current_year_group)) |>
  # transform date columns into date format
  mutate(across(c(date_of_birth, start_date), as_date))

# check data types
glimpse(pupils)

Rows: 163
Columns: 6
$ pupil_id           <chr> "Pupil 191", "Pupil 192", "Pupil 193", "Pupil 194",…
$ date_of_birth      <date> 2022-06-02, 2022-07-06, 2022-08-11, 2022-06-14, 20…
$ current_class      <chr> "Starter group", "Starter group", "Starter group", …
$ current_year_group <int> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, …
$ pupil_name         <chr> "Noor Verbeek", "Daan Veldman", "Isa Van Loon", "Fl…
$ start_date         <date> 2026-06-02, 2026-07-06, 2026-08-31, 2026-06-17, 20…


#### Dataset: UvA_testresults_math.xlsx

- Column names translated from Dutch to English.
- `pupil_number` converted from text to integer
- `score` transformed to a proportion from percentages
- Five exact duplicate rows were removed.
- Linked to pupils dataset by name (lowercase, whitespace removed), because the ID formats differ between files.
- 78 pupils have maths results. 85 don't have the results, 84 are in year groups 0–3, which were not tested.
- 13 names in the maths file have no match in the pupil file (likely former pupils?).
- transformed into long and wide format

In [29]:
# UvA_testresults_math.xlsx
math_raw <- suppressWarnings(read_excel(
  here("data/raw/UvA_testresults_math.xlsx"),
  sheet = "Blad1",
  # read blok and toetsdoel as text
  col_types = c("text", "numeric", "text", "text", "numeric",
                "numeric", "guess", "numeric", "guess", "numeric")
))

# clean Math results
math <- math_raw |>
  # rename columns to English
  rename(
    pupil = leerling,
    pupil_number = Leerlingnummer,
    block = blok,
    test_goal = toetsdoel,
    n_correct = `Vragen goed`,
    n_questions = `Aantal vragen`,
    score  = score,
    passed = behaald,
    date = datum,
    cohort = Cohort
  ) |>
  # turn datum into a date
  mutate(date = as_date(date))

# check the dataset
glimpse(math)

# missing values per column
colSums(is.na(math))

# did some kids retake the tests?
math |>
  count(pupil_number, test_goal) |>
  filter(n > 1)
# yes they did


Rows: 7,190
Columns: 10
$ pupil        <chr> "Eva Aksoy", "Eva Aksoy", "Eva Aksoy", "Eva Aksoy", "Eva …
$ pupil_number <dbl> 7, 7, 7, 7, 7, 19, 19, 19, 19, 19, 5, 5, 5, 5, 5, 10, 10,…
$ block        <chr> "3.2", "3.2", "3.2", "3.2", "3.2", "3.2", "3.2", "3.2", "…
$ test_goal    <chr> "3.2.1", "3.2.2", "3.2.3", "3.2.4", "3.2.5", "3.2.1", "3.…
$ n_correct    <dbl> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, N…
$ n_questions  <dbl> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, N…
$ score        <dbl> 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.…
$ passed       <dbl> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, …
$ date         <date> 2022-10-01, 2022-10-01, 2022-10-01, 2022-10-01, 2022-10-…
$ cohort       <dbl> 2028, 2028, 2028, 2028, 2028, 2028, 2028, 2028, 2028, 202…


pupil pupil_number        block    test_goal    n_correct  n_questions 
           0            0            0            0         1913         1603 
       score       passed         date       cohort 
           0            0         1831            0

pupil_number,test_goal,n
<dbl>,<chr>,<int>
8,3.5.1,2
8,3.5.2,2
8,3.5.3,2
8,3.5.4,2
8,3.5.5,2
16,3.2.1,2
16,3.2.2,2
16,3.2.3,2
16,3.2.4,2


### Merge Pupils and Math results

In [30]:

# Long format----------------------------------------------------------

# Make a clean name key in both files (lowercase, no extra spaces)
pupils <- pupils |> mutate(name_key = str_to_lower(str_squish(pupil_name)))
math <- math |> mutate(name_key = str_to_lower(str_squish(pupil)))


math_long <- math |>
  left_join(
    pupils |> select(-pupil_name), # drop duplicate name column
    by = join_by(name_key),
    relationship = "many-to-one" # stop if a name matches twice
  )

head(math_long)

#pupils with no math results
math_long |>
  filter(is.na(pupil_id)) |>
 distinct(name_key) |>
 nrow() # 13 pupils with no math results


pupil,pupil_number,block,test_goal,n_correct,n_questions,score,passed,date,cohort,name_key,pupil_id,date_of_birth,current_class,current_year_group,start_date
<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<date>,<dbl>,<chr>,<chr>,<date>,<chr>,<int>,<date>
Eva Aksoy,7,3.2,3.2.1,NA,NA,1,1,2022-10-01,2028,eva aksoy,Pupil 015,2016-07-02,Group 7,7,2020-07-02
Eva Aksoy,7,3.2,3.2.2,NA,NA,1,1,2022-10-01,2028,eva aksoy,Pupil 015,2016-07-02,Group 7,7,2020-07-02
Eva Aksoy,7,3.2,3.2.3,NA,NA,1,1,2022-10-01,2028,eva aksoy,Pupil 015,2016-07-02,Group 7,7,2020-07-02
Eva Aksoy,7,3.2,3.2.4,NA,NA,1,1,2022-10-01,2028,eva aksoy,Pupil 015,2016-07-02,Group 7,7,2020-07-02
Eva Aksoy,7,3.2,3.2.5,NA,NA,1,1,2022-10-01,2028,eva aksoy,Pupil 015,2016-07-02,Group 7,7,2020-07-02
Zeynep Van Dam,19,3.2,3.2.1,NA,NA,1,1,2022-10-01,2028,zeynep van dam,NA,NA,NA,NA,NA


[1] 13

In [ ]:
# Wide format ----------------------------------------------------------
# remove exact duplicate rows
math <- math |>
  distinct()

# attempts per pupil and test goal
math_attempts <- math |>
  arrange(name_key, test_goal, date) |>
  mutate(attempt    = row_number(),
         n_attempts = n(),
         .by = c(name_key, test_goal))

# pupil-level info from math
math_pupil_info <- math |>
  summarise(
    pupil_number = first(pupil_number),
    cohort = first(cohort),
    n_numbers = n_distinct(pupil_number),
    n_cohorts = n_distinct(cohort),
    .by = name_key
  )

# summary per pupil and test goal
math_summary <- math_attempts |>
  summarise(
    n_questions = first(n_questions),
    retake = as.integer(n() > 1),
    first_date = first(date),
    first_n_correct = first(n_correct),
    first_score = first(score),
    first_passed = first(passed),
    final_date = last(date),
    final_n_correct = last(n_correct),
    final_score = last(score),
    final_passed = last(passed),
    .by = c(name_key, test_goal)
  )

# check: retakes by first-attempt result
cat("\n--- Retakes by first-attempt result (retake: 1 = yes, first_passed: 1 = passed) ---\n")
math_summary |>
  count(retake, first_passed) |>
  print()

# summary per pupil across all test goals
math_pupil_summary <- math_summary |>
  summarise(
    math_n_goals = n(),
    math_n_passed = sum(final_passed, na.rm = TRUE),
    math_mean_score = mean(final_score, na.rm = TRUE),
    math_n_retakes = sum(retake, na.rm = TRUE),
    math_first_test = min(first_date, na.rm = TRUE),
    math_last_test = max(final_date, na.rm = TRUE),
    .by = name_key
  )

# one block of columns per test goal
math_wide <- math_summary |>
  arrange(test_goal) |>
  pivot_wider(
    id_cols     = name_key,
    names_from  = test_goal,
    values_from = c(n_questions, retake,
                    first_date, first_n_correct, first_score, first_passed,
                    final_date, final_n_correct, final_score, final_passed),
    names_glue  = "math_{test_goal}_{.value}",   # e.g. math_3.5.2_final_score
    names_vary  = "slowest"                      # keep each goal's columns together
  )

# all pupil information in one row per pupil
pupils_wide <- pupils |>
  left_join(math_pupil_info,    by = join_by(name_key), relationship = "one-to-one") |>
  left_join(math_pupil_summary, by = join_by(name_key), relationship = "one-to-one") |>
  left_join(math_wide,          by = join_by(name_key), relationship = "one-to-one")
head(pupils_wide)

# Pupils with math results (year groups 4-7)
cat("\n--- Pupils with math results per year group ---\n")
pupils_wide |>
  filter(!is.na(math_n_goals)) |>
  count(current_year_group) |>
  print()


--- Rows by number of attempts per pupil and test goal ---
# A tibble: 2 × 2
  n_attempts     n
       <int> <int>
1          1  7133
2          2    52

--- Retakes by first-attempt result (retake: 1 = yes, first_passed: 1 = passed) ---
# A tibble: 4 × 3
  retake first_passed     n
   <int>        <dbl> <int>
1      0            0  2182
2      0            1  4951
3      1            0    22
4      1            1     4


pupil_id,date_of_birth,current_class,current_year_group,pupil_name,start_date,name_key,pupil_number,cohort,n_numbers,⋯,math_9.2.4_n_questions,math_9.2.4_retake,math_9.2.4_first_date,math_9.2.4_first_n_correct,math_9.2.4_first_score,math_9.2.4_first_passed,math_9.2.4_final_date,math_9.2.4_final_n_correct,math_9.2.4_final_score,math_9.2.4_final_passed
<chr>,<date>,<chr>,<int>,<chr>,<date>,<chr>,<dbl>,<dbl>,<int>,⋯,<dbl>,<int>,<date>,<dbl>,<dbl>,<dbl>,<date>,<dbl>,<dbl>,<dbl>
Pupil 191,2022-06-02,Starter group,0,Noor Verbeek,2026-06-02,noor verbeek,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
Pupil 192,2022-07-06,Starter group,0,Daan Veldman,2026-07-06,daan veldman,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
Pupil 193,2022-08-11,Starter group,0,Isa Van Loon,2026-08-31,isa van loon,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
Pupil 194,2022-06-14,Starter group,0,Floris De Ruiter,2026-06-17,floris de ruiter,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
Pupil 195,2022-08-11,Starter group,0,Lena Smeets,2026-08-31,lena smeets,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
Pupil 196,2022-08-13,Starter group,0,Niels Van Essen,2026-08-31,niels van essen,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA



--- Pupils with math results per year group ---
# A tibble: 5 × 2
  current_year_group     n
               <int> <int>
1                  3     1
2                  4    30
3                  5    15
4                  6    14
5                  7    18


#### Dataset: Observaties_groep_1_2.xlsx


- Column names translated to English
- empty cells and "NA" text set to missing.
- The undocumented column `Leercode` was kept as `objective_code`
- Issues in Scores: 19 values outside 0–4 (one 5 and two-digit values such as 22, 33, 36) were treated as entry errors and set to missing -> original values kept in `score_raw`.
- Subject and domain abbreviations were linked to the full names
- Domains renamed between school years were merged: 
    Boekoriëntatie → Boekoriëntatie en verhaalbegrip;
    Getallen and Tellen en getalbegrip → Tellen en getallen;
    Spel/Spelen within the subject Play.
- English translations added for subject, domain, learning path, subject type and class group, using the translation sheet.
- `item_code` created from `learning_path_code`, or `objective_code` when it was missing -> 104 rows have no item code.
- Exact duplicate rows were removed after recoding.
- Missing or unclear domains were filled from the item code where that code belongs to a single known domain (in total 4,590 rows)
- Remaining without a domain: 434 rows with unresolved codes (GV, SR, TS, EP) and 1,069 rows with no domain (~1% of all rows).

- The domains change across years: several gross-motor and world-orientation domains appear in only some years, and the subject Child understanding exists only from 2024/2025 onwards.
- 48 observed names have no match in the pupil file. 34 were first observed in 2019–2022 (likely former pupils??). Of the 14 first observed in 2023 or later.

In [52]:
# Load Observations group 1_2
obs_raw <- read_excel(
  here("data/raw/Observaties_groep_1_2.xlsx"),
  na = c("", "NA"),   # treat empty cells and "NA" text as missing
  col_types = c(rep("text", 7), rep("guess", 5))
) |>
  mutate(Score = as.numeric(Score))

# Load translations sheet ----
translations <- read_excel(
  here("data/raw/Observaties_groep_1_2.xlsx"),
  sheet = "Translations",
  skip = 2
) |>
  rename(category = Category, dutch = `Dutch term`, english = `English translation`) |>
  distinct(category, dutch, .keep_all = TRUE)   # table has some duplicate rows

In [53]:
# Look up the English term for a Dutch value (NA if not in the translations sheet)
translate <- function(x, cat) {
  lookup <- filter(translations, category == cat)
  lookup$english[match(x, lookup$dutch)]
}

# Domain codes that could not be matched to a full name
unclear_domains <- c("OV", "PKS", "VKS", "GV", "SR", "TS", "EP")

# Clean Observations group 1/2 ----
obs <- obs_raw |>
  # Rename columns to English
  rename(
    objective_code     = Leercode,
    subject            = Vak,
    domain             = Domein,
    learning_path      = Leerpad,
    learning_path_code = Leerpadcode,
    pupil              = Leerling,
    score              = Score,
    date               = Datum,
    period             = Periode,
    school_year        = Jaargang,
    subject_type       = Vaktype,
    class_group        = Groep
  ) |>
  # Set types
  mutate(
    score  = as.integer(score),
    period = as.integer(period),
    date   = as_date(date)
  ) |>
  # Keep original score, set scores outside 0-4 to missing
  mutate(
    score_raw = score,
    score     = if_else(score %in% 0:4, score, NA)
  ) |>
  # Recode subject abbreviations to full Dutch names
  mutate(subject = case_match(
    subject,
    "BV" ~ "Burgerschapsvorming",
    "FL" ~ "Filosofie",
    "HT" ~ "Handvaardigheid en tekenen",
    "PK" ~ "Podiumkunsten",
    .default = subject
  )) |>
  # Recode domain abbreviations to full Dutch names (unclear codes stay as they are)
  mutate(domain = case_match(
    domain,
    "JA"         ~ "Jezelf en de ander",
    "SL"         ~ "De samenleving",
    "HT"         ~ "Handvaardigheid en tekenen",
    "KM"         ~ "Kleine motoriek",
    "B"          ~ "Balanceren",
    "KK"         ~ "Klimmen",                           # codes start with KL
    "R"          ~ "Over de kop gaan",                  # codes start with OK
    "IT"         ~ "Interactief taalgebruik",
    "BG"         ~ "Boekoriëntatie en verhaalbegrip",   # codes start with BV
    "MK"         ~ "Meetkunde",
    "MT"         ~ "Meten",
    "TG"         ~ "Tellen en getallen",
    "PK"         ~ "Podiumkunsten",
    c("F", "FL") ~ "Filosofie",
    "S"          ~ "Spel",
    "SP"         ~ "Spelen",
    "BT"         ~ "Betrokkenheid",
    "IB"         ~ "Impulsbeheersing",
    "IV"         ~ "Inlevingsvermogen",
    "SA"         ~ "Sociale autonomie",
    "SF"         ~ "Sociale flexibiliteit",
    "SI"         ~ "Sociaal initiatief",
    "TZ"         ~ "Taakgerichtheid en zelfstandigheid",
    "WB"         ~ "Welbevinden",
    "K"          ~ "Kennis",
    "PDM"        ~ "Planten, dieren en de mens",
    "RU"         ~ "Ruimte om je heen",
    "T"          ~ "Tijd",
    .default = domain
  )) |>
  # Merge domain names that were renamed between school years
  mutate(domain = case_when(
    domain == "Boekoriëntatie"                         ~ "Boekoriëntatie en verhaalbegrip",
    domain %in% c("Getallen", "Tellen en getalbegrip") ~ "Tellen en getallen",
    subject == "Spel" & domain == "Spelen"             ~ "Spel",
    .default = domain
  )) |>
  # Move FL_ codes from learning_path to learning_path_code, recode SP
  mutate(
    learning_path_code = case_when(
      str_starts(learning_path, "FL_") & is.na(learning_path_code) ~ learning_path,
      .default = learning_path_code
    ),
    learning_path = case_when(
      learning_path == "SP"            ~ "Spel",
      str_starts(learning_path, "FL_") ~ NA,
      .default = learning_path
    )
  ) |>
  # Add English translations
  mutate(
    subject_en       = translate(subject, "Vak"),
    domain_en        = translate(domain, "Domein"),
    learning_path_en = translate(learning_path, "Leerpad"),
    subject_type_en  = translate(subject_type, "Vaktype"),
    class_group_en   = translate(class_group, "Groep")
  ) |>
  # Item identifier: learning_path_code, or objective_code when it is missing
  mutate(item_code = coalesce(learning_path_code, objective_code)) |>
  # First year of the school year ("2019/2020" -> 2019)
  mutate(school_year_start = as.integer(str_sub(school_year, 1, 4))) |>
  # Clean name key for joining
  mutate(name_key = str_to_lower(str_squish(pupil))) |>
  # Remove exact duplicate rows (after recoding, so code vs full name duplicates go too)
  distinct()

# Fill domains from item code ----
# Item codes that belong to exactly one known domain
code_domain <- obs |>
  filter(!is.na(item_code), !is.na(domain_en)) |>
  distinct(item_code, domain, domain_en) |>
  filter(n() == 1, .by = item_code) |>
  rename(domain_fill = domain, domain_en_fill = domain_en)

# Fill missing or unclear domains, keep a flag for filled rows
obs <- obs |>
  left_join(code_domain, by = join_by(item_code), relationship = "many-to-one") |>
  mutate(
    domain_filled = (is.na(domain) | domain %in% unclear_domains) & !is.na(domain_fill),
    domain        = if_else(domain_filled, domain_fill, domain),
    domain_en     = if_else(domain_filled, domain_en_fill, domain_en)
  ) |>
  select(-domain_fill, -domain_en_fill)

#### Join observations data to pupils wide dataset

In [59]:
# domains present in all school years
stable_domains <- c(
  "Auditory development", "Book orientation and story comprehension",
  "Interactive language use", "Visual development",                    # Language
  "Counting and numbers", "Geometry", "Measurement",                   # Mathematics
  "Empathy", "Engagement", "Impulse control", "Social autonomy",
  "Social flexibility", "Social initiative",
  "Task focus and independence", "Well-being"                          # Social-emotional
)

# observations used for the wide file
obs_scored <- obs |>
  filter(!is.na(score), subject_en != "Child understanding") |>
  # scale maximum per item: 4 if the item ever has a 4, otherwise 3
  mutate(item_max = if_else(any(score == 4), 4L, 3L), .by = item_code) |>
  # score rescaled to 0-1
  mutate(score_scaled = score / item_max) |>
  # short column-friendly names ("Fine motor skills" -> "fine_motor_skills")
  mutate(
    subject_key = str_to_lower(str_replace_all(subject_en, "[^A-Za-z]+", "_")),
    domain_key  = str_to_lower(str_replace_all(domain_en,  "[^A-Za-z]+", "_"))
  )

# per pupil: overall observation info
obs_pupil_info <- obs_scored |>
  summarise(
    obs_n_total     = n(),
    obs_n_years     = n_distinct(school_year),
    obs_first_date  = min(date),
    obs_last_date   = max(date),
    obs_mean_scaled = mean(score_scaled),
    .by = name_key
  )

# per pupil and subject: mean scaled score and number of observations
obs_subject_wide <- obs_scored |>
  summarise(mean = mean(score_scaled), n = n(), .by = c(name_key, subject_key)) |>
  arrange(subject_key) |>
  pivot_wider(
    id_cols     = name_key,
    names_from  = subject_key,
    values_from = c(mean, n),
    names_glue  = "obs_{subject_key}_{.value}",   # e.g. obs_mathematics_mean
    names_vary  = "slowest"
  )

# per pupil and stable domain: mean scaled score and number of observations
obs_domain_wide <- obs_scored |>
  filter(subject_en %in% c("Language", "Mathematics", "Social-emotional development"),
         domain_en %in% stable_domains) |>
  summarise(mean = mean(score_scaled), n = n(), .by = c(name_key, domain_key)) |>
  arrange(domain_key) |>
  pivot_wider(
    id_cols     = name_key,
    names_from  = domain_key,
    values_from = c(mean, n),
    names_glue  = "obs_dom_{domain_key}_{.value}",   # e.g. obs_dom_geometry_mean
    names_vary  = "slowest"
  )

# add to pupils_wide (keeps pupils without observations)
pupils_wide <- pupils_wide |>
  left_join(obs_pupil_info,   by = join_by(name_key), relationship = "one-to-one") |>
  left_join(obs_subject_wide, by = join_by(name_key), relationship = "one-to-one") |>
  left_join(obs_domain_wide,  by = join_by(name_key), relationship = "one-to-one")

# first rows of the final table
head(pupils_wide)

pupil_id,date_of_birth,current_class,current_year_group,pupil_name,start_date,name_key,pupil_number,cohort,n_numbers,⋯,obs_dom_social_flexibility_mean.y,obs_dom_social_flexibility_n.y,obs_dom_social_initiative_mean.y,obs_dom_social_initiative_n.y,obs_dom_task_focus_and_independence_mean.y,obs_dom_task_focus_and_independence_n.y,obs_dom_visual_development_mean.y,obs_dom_visual_development_n.y,obs_dom_well_being_mean.y,obs_dom_well_being_n.y
<chr>,<date>,<chr>,<int>,<chr>,<date>,<chr>,<dbl>,<dbl>,<int>,⋯,<dbl>,<int>,<dbl>,<int>,<dbl>,<int>,<dbl>,<int>,<dbl>,<int>
Pupil 191,2022-06-02,Starter group,0,Noor Verbeek,2026-06-02,noor verbeek,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
Pupil 192,2022-07-06,Starter group,0,Daan Veldman,2026-07-06,daan veldman,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
Pupil 193,2022-08-11,Starter group,0,Isa Van Loon,2026-08-31,isa van loon,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
Pupil 194,2022-06-14,Starter group,0,Floris De Ruiter,2026-06-17,floris de ruiter,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
Pupil 195,2022-08-11,Starter group,0,Lena Smeets,2026-08-31,lena smeets,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
Pupil 196,2022-08-13,Starter group,0,Niels Van Essen,2026-08-31,niels van essen,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


## Export pupils wide dataset to xlsx

In [65]:
# remove columns that are empty for all pupils
n_before <- ncol(pupils_wide)

pupils_wide <- pupils_wide |>
  select(where(\(x) !all(is.na(x))))

cat("\n--- Empty columns removed ---\n")
print(n_before - ncol(pupils_wide))


--- Empty columns removed ---
[1] 79


In [67]:
wb <- createWorkbook()
addWorksheet(wb, "pupils_wide")
writeData(wb, "pupils_wide", pupils_wide, withFilter = TRUE)
# keep header row and first 5 columns visible when scrolling
freezePane(wb, "pupils_wide", firstActiveRow = 2, firstActiveCol = 6)
saveWorkbook(wb, here("data/processed/pupils_wide.xlsx"), overwrite = TRUE)